# TP 9 – Votre premier stream (35 min)

**Objectif** : suivre en direct le chiffre d'affaires et le nombre de courses par arrondissement.

**Réussi si** : le tableau se met à jour à chaque micro-batch.

**Avant de commencer**, dans un terminal à la racine du dépôt :

```bash
python scripts/stream_simulator.py courses --reset          # en direct : un fichier toutes les 2 secondes
# ou, si vous êtes en retard / en mode dégradé :
python scripts/stream_simulator.py courses --all --reset    # tout d'un coup
```

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("TP").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")     # petit volume : 200 partitions par défaut, c'est trop

### 1. Lire le dossier `stream/courses` en continu

Un stream exige un **schéma explicite** : prenez celui de `data/trips.parquet` (`spark.read.parquet(...).schema`).
Indices : `spark.readStream.schema(schema).option("maxFilesPerTrigger", 2).json("stream/courses")`, puis `isStreaming`.

In [ ]:
schema = ...
courses = ...
print(courses.isStreaming)

### 2. Jointure stream-statique avec les zones

Le référentiel `data/taxi_zone_lookup.csv` est un DataFrame **statique** ordinaire. Ajoutez la colonne `Borough`.

In [ ]:
zones = ...
enrichies = ...

### 3. Chiffre d'affaires et nombre de courses par `Borough`

In [ ]:
par_borough = ...

### 4. Démarrer la requête : `outputMode("complete")`, `format("console")`, `trigger(processingTime="5 seconds")`, un `checkpointLocation`

Laissez tourner 60 secondes : `q.awaitTermination(60)`, puis `q.stop()`. Les tableaux s'affichent sous la cellule, un par micro-batch.

⚠ Le checkpoint est la **mémoire** de la requête : si vous relancez après un `--reset` du simulateur, supprimez-le d'abord (première ligne de la cellule), sinon la requête reprend l'ancien état et ignore les nouveaux fichiers.

In [ ]:
import shutil
shutil.rmtree("out/checkpoints/tp9", ignore_errors=True)   # repartir de zéro

q = (par_borough.writeStream
     ...
     .option("checkpointLocation", "out/checkpoints/tp9")
     .start())
q.awaitTermination(60)
q.stop()

### 5. Le suivi : `q.lastProgress`, `q.recentProgress`

Combien de micro-batches ? Combien de lignes par micro-batch (`numInputRows`) ?

In [ ]:
...

### Bonus – Le même calcul en **batch** (`spark.read`) sur les mêmes fichiers : obtenez-vous les mêmes comptes ?

Si le stream en a moins : comparez le rythme du simulateur (1 fichier toutes les 2 s) et `maxFilesPerTrigger` (2 fichiers par micro-batch de 5 s). Le stream a-t-il pris du retard ? Que se passe-t-il si on le laisse tourner plus longtemps ?

In [ ]:
...